In [2]:
import sys
import yaml
import argparse
import numpy as np
import igraph as ig
import networkx as nx
import matplotlib.pyplot as plt
from qlgates.config import Config
from qlgates.run_dynamics import propagate_state, build_unitary, bell_state
from qlgates.qlgraphs import qldit, cart_qldit, compute_eigen_decomposition
from qlgates.gates import get_Vg
from core.contraction import minimal_quotient
from core.graph_generation import generate_quantum_like_bit

In [3]:
print(sys.executable)   # should show your conda env path, not system python

"""Load YAML and merge into Config dataclass."""
with open("../configs/run1.yaml") as f:
    overrides = yaml.safe_load(f)  # plain dict from YAML

# These are computed in __post_init__, never let YAML set them
overrides.pop("l", None)
overrides.pop("lp", None)

# Create Config instance with merged parameters
cfg = Config(**overrides)

/Users/sahadebadrita/opt/anaconda3/envs/graphs/bin/python


In [4]:
#Checking Will's package
qlbit_1p, info = generate_quantum_like_bit(cfg.n,cfg.k,cfg.l)
qlbit_2p, info = generate_quantum_like_bit(cfg.n,cfg.k,cfg.lp)
qlbit_3p, info = generate_quantum_like_bit(cfg.n,cfg.k,cfg.l)

In [5]:
qlbit_1pmin,info_min = minimal_quotient((qlbit_1p,info))
qlbit_2pmin,info_min = minimal_quotient((qlbit_2p,info))
qlbit_3pmin,info_min = minimal_quotient((qlbit_3p,info))

In [6]:
min_cartpdt12 = cart_qldit(qlbit_2pmin,qlbit_3pmin)
min_cartpdt = cart_qldit(qlbit_1pmin,min_cartpdt12)
print(min_cartpdt.shape)

(8, 8)


In [7]:
e_min_cartpdt, v_min_cartpdt = np.linalg.eigh(min_cartpdt)
print(e_min_cartpdt[-5:])

[28. 32. 32. 32. 36.]


Test single QL-bit states in min rep

In [8]:
e_qlbit1_min, v_qlbit1_min = np.linalg.eigh(qlbit_1pmin)
print(e_qlbit1_min)
print(v_qlbit1_min)

[ 8. 12.]
[[-0.70710678  0.70710678]
 [ 0.70710678  0.70710678]]


In [10]:
H = get_Vg('H',theta=None,U=None)
X = get_Vg('x',theta=None,U=None)

Ux = H @ X @ H
print(Ux @ v_qlbit1_min)
print(Ux @ qlbit_1pmin @ Ux.T.conj())

[[-0.70710678+0.j  0.70710678+0.j]
 [-0.70710678+0.j -0.70710678+0.j]]
[[10.+0.j -2.+0.j]
 [-2.+0.j 10.+0.j]]


In [11]:
H = get_Vg('H',theta=None,U=None)
Z = get_Vg('z',theta=None,U=None)

Uz = H @ Z @ H
print(Uz @ v_qlbit1_min)
print(Uz @ qlbit_1pmin @ Uz.T.conj())

[[ 0.70710678+0.j  0.70710678+0.j]
 [-0.70710678+0.j  0.70710678+0.j]]
[[10.+0.j  2.+0.j]
 [ 2.+0.j 10.+0.j]]


In [13]:
H = get_Vg('H',theta=None,U=None)
RX = get_Vg('Rx',theta=1.57,U=None)

URx = H @ RX @ H
print(URx @ v_qlbit1_min)
print(URx @ qlbit_1pmin @ URx.T.conj())

[[-0.50019904+0.49980088j  0.50019904-0.49980088j]
 [ 0.50019904+0.49980088j  0.50019904+0.49980088j]]
[[1.00000000e+01+0.j         1.59265342e-03-1.99999937j]
 [1.59265342e-03+1.99999937j 1.00000000e+01+0.j        ]]


Basis change for 2 ql-bits

In [20]:
e_qlbit2_min, v_qlbit2_min = np.linalg.eigh(qlbit_2pmin)
print(e_qlbit2_min)
print(v_qlbit2_min)

[ 8. 12.]
[[-0.70710678  0.70710678]
 [ 0.70710678  0.70710678]]


In [19]:
e_min_cartpdt12, v_min_cartpdt12 = np.linalg.eigh(min_cartpdt12)

UCB2 = np.kron(H, H)

print(e_min_cartpdt12)
print(v_min_cartpdt12)
print(np.real(UCB2 @ v_min_cartpdt12))

[16. 20. 20. 24.]
[[ 0.5         0.64628163  0.28691471  0.5       ]
 [-0.5        -0.28691471  0.64628163  0.5       ]
 [-0.5         0.28691471 -0.64628163  0.5       ]
 [ 0.5        -0.64628163 -0.28691471  0.5       ]]
[[-7.49400542e-16 -1.11022302e-15  1.19348975e-15  1.00000000e+00]
 [ 4.71844785e-16  9.33196342e-01 -3.59366926e-01  1.66533454e-15]
 [-6.66133815e-16  3.59366926e-01  9.33196342e-01 -7.77156117e-16]
 [ 1.00000000e+00 -1.66533454e-16  8.60422844e-16  9.99200722e-16]]


In [24]:
CNOT = np.array([
    [1, 0, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0],
    [0, 1, 0, 0]
])

In [28]:
UCNOT = UCB2 @ CNOT @ UCB2

print((UCNOT @ v_min_cartpdt12).real - v_min_cartpdt12.real)

[[-2.22044605e-16 -3.33066907e-16 -1.11022302e-16 -2.22044605e-16]
 [ 2.22044605e-16  1.11022302e-16 -3.33066907e-16 -2.22044605e-16]
 [ 1.00000000e+00 -9.33196342e-01  3.59366926e-01 -8.88178420e-16]
 [-1.00000000e+00  9.33196342e-01 -3.59366926e-01  4.44089210e-16]]
